# 052 Blend Add051 Trimmed Check

Trimmed blend/correlation check based on the 050 implementation.

Purpose:
- Add 051 `shared001_updated_v2_realmlp_pytorch_as_is`.
- Keep correlation diagnostics.
- Reduce runtime by omitting models that had zero/nearly-zero weight in the 050 best safe blend, or were superseded by later assets.
- Disable in-sample meta screening by default; safe blend diagnostics remain enabled.

Expected role:
- Determine whether 051 has useful safe-blend weight with 050 / 049 / 040 / 038 / 033 / 045.
- If 051 receives meaningful weight and improves CV/Public LB, consider a later GPU LogReg stacker add051.


In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260615_052_blend_add051_trimmed_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
# Fallback for the account spelling used in earlier runs.
NPY_DIR_FALLBACK = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

# 050 took 6h+ partly because it ran in-sample meta models for many blend sets.
# For 052, keep the correlation/safe-blend diagnostics and disable meta screening by default.
RUN_IN_SAMPLE_META = False

MODEL_SPECS = [{'key': '015_realmlp_seed42',
  'exp_id': 'exp_20260605_015_shared001_realmlp_as_is',
  'family': 'RealMLP',
  'role': 'stable_single_realmlp_backup_positive_weight_in_050',
  'oof': 'oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy',
  'pred': 'pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy',
  'cv': 0.9681693449222352,
  'public_lb': 0.96977},
 {'key': '026_realmlp_v5',
  'exp_id': 'exp_20260608_026_realmlp_v5_as_is',
  'family': 'RealMLP',
  'role': 'realmlp_v5_single_model_positive_weight_in_050',
  'oof': 'oof_exp_20260608_026_realmlp_v5_as_is_proba.npy',
  'pred': 'pred_exp_20260608_026_realmlp_v5_as_is_proba.npy',
  'cv': 0.9690389777981231,
  'public_lb': 0.96979},
 {'key': '027_blend_add026',
  'exp_id': 'exp_20260608_027_blend_add026_realmlp_v5_check',
  'family': 'Blend',
  'role': 'previous_cv_trusted_slot_positive_weight_in_050',
  'oof': 'oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy',
  'pred': 'pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy',
  'cv': 0.9695425457477898,
  'public_lb': 0.96975},
 {'key': '028_cat_v3',
  'exp_id': 'exp_20260608_028_cat_v3_as_is',
  'family': 'CatBoost',
  'role': 'catboost_v3_positive_weight_in_050',
  'oof': 'oof_exp_20260608_028_cat_v3_as_is_proba.npy',
  'pred': 'pred_exp_20260608_028_cat_v3_as_is_proba.npy',
  'cv': 0.9688146470512534,
  'public_lb': 0.96969},
 {'key': '029_blend_add028',
  'exp_id': 'exp_20260608_029_blend_add028_cat_v3_check',
  'family': 'Blend',
  'role': 'previous_best_backup_positive_weight_in_050',
  'oof': 'oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy',
  'pred': 'pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy',
  'cv': 0.9700023026377228,
  'public_lb': 0.970036},
 {'key': '031_blend_add030',
  'exp_id': 'exp_20260608_031_blend_add030_ovr_xgb_check',
  'family': 'Blend',
  'role': 'public_confirmed_before_033_positive_weight_in_050',
  'oof': 'oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy',
  'pred': 'pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy',
  'cv': 0.9700333620193362,
  'public_lb': 0.97043},
 {'key': '032_ovr_tabm',
  'exp_id': 'exp_20260609_032_ovr_tabm_as_is',
  'family': 'TabM',
  'role': 'tabm_ovr_material_positive_weight_in_050',
  'oof': 'oof_exp_20260609_032_ovr_tabm_as_is_proba.npy',
  'pred': 'pred_exp_20260609_032_ovr_tabm_as_is_proba.npy',
  'cv': 0.9610105651284856,
  'public_lb': 0.96106,
  'tuned_cv': 0.9686168281485955,
  'public_lb_biased': 0.96895},
 {'key': '033_blend_add032',
  'exp_id': 'exp_20260609_033_blend_add032_tabm_check',
  'family': 'Blend',
  'role': 'private_diversity_hedge_positive_weight_in_050',
  'oof': 'oof_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy',
  'pred': 'pred_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy',
  'cv': 0.9700400336552478,
  'public_lb': 0.97043},
 {'key': '035_shared001_updated',
  'exp_id': 'exp_20260610_035_shared001_updated_realmlp_pytorch_as_is',
  'family': 'RealMLP',
  'role': 'shared001_updated_positive_weight_in_050_reference_for_051',
  'oof': 'oof_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy',
  'pred': 'pred_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy',
  'cv': 0.9687410900866934,
  'public_lb': 0.97012},
 {'key': '036_blend_add035',
  'exp_id': 'exp_20260610_036_blend_add035_shared001_check',
  'family': 'Blend',
  'role': 'strong_positive_weight_in_050',
  'oof': 'oof_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy',
  'pred': 'pred_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy',
  'cv': 0.9700727843277738,
  'public_lb': 0.97023},
 {'key': '038_gpu_logreg_add037',
  'exp_id': 'exp_20260610_038_gpu_logreg_stacker_add037_own',
  'family': 'GPU_LogisticRegression_Stacker',
  'role': 'public_lb_backup_keep_even_zero_weight_in_full_050',
  'oof': 'oof_exp_20260610_038_gpu_logreg_stacker_add037_own_proba.npy',
  'pred': 'pred_exp_20260610_038_gpu_logreg_stacker_add037_own_proba.npy',
  'cv': 0.9701353365798572,
  'public_lb': 0.9706},
 {'key': '039_cat_v3_seed2026_qbin_variant',
  'exp_id': 'exp_20260611_039_cat_v3_seed2026_qbin_variant',
  'family': 'CatBoost',
  'role': 'large_positive_weight_in_050',
  'oof': 'oof_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy',
  'pred': 'pred_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy',
  'cv': 0.9687053023092482,
  'public_lb': 0.96984},
 {'key': '040_gpu_logreg_add039',
  'exp_id': 'exp_20260611_040_gpu_logreg_stacker_add039_own',
  'family': 'GPU_LogisticRegression_Stacker',
  'role': 'current_best_public_confirmed_positive_weight_in_050',
  'oof': 'oof_exp_20260611_040_gpu_logreg_stacker_add039_own_proba.npy',
  'pred': 'pred_exp_20260611_040_gpu_logreg_stacker_add039_own_proba.npy',
  'cv': 0.9702017877238539,
  'public_lb': 0.97069},
 {'key': '042_realmlp_v5_seed2026_foldvariant',
  'exp_id': 'exp_20260611_042_realmlp_v5_seed2026_foldvariant',
  'family': 'RealMLP',
  'role': 'small_positive_weight_in_050_realmlp_variant',
  'oof': 'oof_exp_20260611_042_realmlp_v5_seed2026_foldvariant_proba.npy',
  'pred': 'pred_exp_20260611_042_realmlp_v5_seed2026_foldvariant_proba.npy',
  'cv': 0.9689482884928097,
  'public_lb': 0.96943},
 {'key': '044_shared001_updated_seed2027_foldvariant',
  'exp_id': 'exp_20260612_044_shared001_updated_seed2027_foldvariant',
  'family': 'RealMLP',
  'role': 'small_positive_weight_in_050_shared001_variant',
  'oof': 'oof_exp_20260612_044_shared001_updated_seed2027_foldvariant_proba.npy',
  'pred': 'pred_exp_20260612_044_shared001_updated_seed2027_foldvariant_proba.npy',
  'cv': 0.9686035095582338,
  'public_lb': 0.97},
 {'key': '045_gpu_logreg_add044',
  'exp_id': 'exp_20260612_045_gpu_logreg_stacker_add044_own',
  'family': 'GPU_LogisticRegression_Stacker',
  'role': 'cv_strong_stacker_positive_weight_in_050',
  'oof': 'oof_exp_20260612_045_gpu_logreg_stacker_add044_own_proba.npy',
  'pred': 'pred_exp_20260612_045_gpu_logreg_stacker_add044_own_proba.npy',
  'cv': 0.9702445493383917,
  'public_lb': 0.9704,
  'raw_cv': 0.9701651297689362,
  'tuned_cv': 0.9702445493383917},
 {'key': '049_gpu_logreg_add048',
  'exp_id': 'exp_20260614_049_gpu_logreg_stacker_add048_own',
  'family': 'GPU_LogisticRegression_Stacker',
  'role': 'largest_positive_weight_in_050',
  'oof': 'oof_exp_20260614_049_gpu_logreg_stacker_add048_own_proba.npy',
  'pred': 'pred_exp_20260614_049_gpu_logreg_stacker_add048_own_proba.npy',
  'cv': 0.9701958734042008,
  'public_lb': 0.97046,
  'raw_cv': 0.97011955078962,
  'tuned_cv': 0.9701958734042008},
 {'key': '050_blend_add047_048_049_best',
  'exp_id': 'exp_20260614_050_blend_add047_048_049_check',
  'family': 'Blend',
  'role': 'current_cv_best_blend_reference',
  'oof': 'oof_exp_20260614_050_blend_add047_048_049_check_best_blend_proba.npy',
  'pred': 'pred_exp_20260614_050_blend_add047_048_049_check_best_blend_proba.npy',
  'cv': 0.9702769901284838,
  'public_lb': 0.97052},
 {'key': '051_shared001_updated_v2',
  'exp_id': 'exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is',
  'family': 'RealMLP',
  'role': 'new_public_strong_shared001_updated_v2_material',
  'oof': 'oof_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is_proba.npy',
  'pred': 'pred_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is_proba.npy',
  'cv': 0.9689075888054314,
  'public_lb': 0.97055}]

OMITTED_MODEL_NOTES = [{'key': '017_realmlp_bias', 'reason': 'zero weight in 050 best blend and superseded by later RealMLP/shared001 variants'},
 {'key': '018_xgb_shared003', 'reason': 'zero weight in 050 best blend; older XGB material'},
 {'key': '019_blend_best', 'reason': 'zero weight in 050 best blend; old composite'},
 {'key': '020_blend_bias', 'reason': 'zero weight in 050 best blend; old bias blend'},
 {'key': '024_seed_ensemble_blend', 'reason': 'zero weight in 050 best blend; old seed ensemble blend'},
 {'key': '030_ovr_xgb', 'reason': 'zero weight in 050 best blend and superseded by 048/049 path'},
 {'key': '034_gpu_logreg_default', 'reason': 'only about 0.99% weight in 050 full blend; stacker family covered by 040/045/049/050'},
 {'key': '036_gpu_logreg_add035', 'reason': 'only about 0.57% weight in 050 full blend; stacker family covered by 040/045/049/050'},
 {'key': '037_tabicl_v2', 'reason': 'zero weight in 050 best blend and weak standalone'},
 {'key': '047_xgb_multiclass_fe_objective_foldvariant', 'reason': 'zero weight in 050 best blend and failed standalone multiclass XGB'},
 {'key': '048_ovr_xgb_seed2027_fe_foldvariant', 'reason': 'zero direct weight in 050; its signal is represented through 049'}]

TRACK_KEYS = {'051': '051_shared001_updated_v2',
 '050': '050_blend_add047_048_049_best',
 '049': '049_gpu_logreg_add048',
 '045': '045_gpu_logreg_add044',
 '044': '044_shared001_updated_seed2027_foldvariant',
 '042': '042_realmlp_v5_seed2026_foldvariant',
 '040': '040_gpu_logreg_add039',
 '039': '039_cat_v3_seed2026_qbin_variant',
 '038': '038_gpu_logreg_add037',
 '036_blend': '036_blend_add035',
 '035': '035_shared001_updated',
 '033': '033_blend_add032',
 '032': '032_ovr_tabm',
 '031': '031_blend_add030',
 '029': '029_blend_add028',
 '028': '028_cat_v3',
 '026': '026_realmlp_v5',
 '015': '015_realmlp_seed42'}

FOCUS_MODEL_KEYS = {'051': '051_shared001_updated_v2',
 '050': '050_blend_add047_048_049_best',
 '049': '049_gpu_logreg_add048',
 '040': '040_gpu_logreg_add039',
 '038': '038_gpu_logreg_add037',
 '033': '033_blend_add032',
 '045': '045_gpu_logreg_add044',
 '036_blend': '036_blend_add035',
 '039': '039_cat_v3_seed2026_qbin_variant',
 '035': '035_shared001_updated',
 '015': '015_realmlp_seed42'}

# 052 should answer:
# - Does 051 receive meaningful safe-blend weight?
# - Does 051 improve over 050 / 040 / 038 when blended?
# - Does 051 replace older shared001 lineage models 015/035/044?
# - Keep interpretation conservative: use safe blends for submission decisions; in-sample meta is disabled by default.
BLEND_SETS = {'A_051_only': ['051_shared001_updated_v2'],
 'B_050_only': ['050_blend_add047_048_049_best'],
 'C_049_only': ['049_gpu_logreg_add048'],
 'D_040_only': ['040_gpu_logreg_add039'],
 'E_038_only': ['038_gpu_logreg_add037'],
 'F_033_only': ['033_blend_add032'],
 'G_045_only': ['045_gpu_logreg_add044'],
 'H_036_blend_only': ['036_blend_add035'],
 'I_039_only': ['039_cat_v3_seed2026_qbin_variant'],
 'J_035_only': ['035_shared001_updated'],
 'K_015_only': ['015_realmlp_seed42'],
 'L_051_040': ['051_shared001_updated_v2', '040_gpu_logreg_add039'],
 'M_051_038': ['051_shared001_updated_v2', '038_gpu_logreg_add037'],
 'N_051_033': ['051_shared001_updated_v2', '033_blend_add032'],
 'O_051_045': ['051_shared001_updated_v2', '045_gpu_logreg_add044'],
 'P_051_049': ['051_shared001_updated_v2', '049_gpu_logreg_add048'],
 'Q_051_050': ['051_shared001_updated_v2', '050_blend_add047_048_049_best'],
 'R_051_036blend': ['051_shared001_updated_v2', '036_blend_add035'],
 'S_051_039': ['051_shared001_updated_v2', '039_cat_v3_seed2026_qbin_variant'],
 'T_051_040_038': ['051_shared001_updated_v2', '040_gpu_logreg_add039', '038_gpu_logreg_add037'],
 'U_051_049_040_038': ['051_shared001_updated_v2', '049_gpu_logreg_add048', '040_gpu_logreg_add039', '038_gpu_logreg_add037'],
 'V_051_045_040_038': ['051_shared001_updated_v2', '045_gpu_logreg_add044', '040_gpu_logreg_add039', '038_gpu_logreg_add037'],
 'W_051_049_045_040_038': ['051_shared001_updated_v2',
                           '049_gpu_logreg_add048',
                           '045_gpu_logreg_add044',
                           '040_gpu_logreg_add039',
                           '038_gpu_logreg_add037'],
 'X_051_049_045_040_038_033': ['051_shared001_updated_v2',
                               '049_gpu_logreg_add048',
                               '045_gpu_logreg_add044',
                               '040_gpu_logreg_add039',
                               '038_gpu_logreg_add037',
                               '033_blend_add032'],
 'Y_051_050_040_038': ['051_shared001_updated_v2', '050_blend_add047_048_049_best', '040_gpu_logreg_add039', '038_gpu_logreg_add037'],
 'Z01_051_050_049_040_038': ['051_shared001_updated_v2',
                             '050_blend_add047_048_049_best',
                             '049_gpu_logreg_add048',
                             '040_gpu_logreg_add039',
                             '038_gpu_logreg_add037'],
 'Z02_051_050_049_045_040_038_033': ['051_shared001_updated_v2',
                                     '050_blend_add047_048_049_best',
                                     '049_gpu_logreg_add048',
                                     '045_gpu_logreg_add044',
                                     '040_gpu_logreg_add039',
                                     '038_gpu_logreg_add037',
                                     '033_blend_add032'],
 'Z03_shared001_lineage_015_035_051': ['015_realmlp_seed42', '035_shared001_updated', '051_shared001_updated_v2'],
 'Z04_realmlp_family_026_042_044_051': ['026_realmlp_v5',
                                        '042_realmlp_v5_seed2026_foldvariant',
                                        '044_shared001_updated_seed2027_foldvariant',
                                        '051_shared001_updated_v2'],
 'Z05_051_plus_realmlp_and_core': ['051_shared001_updated_v2',
                                   '026_realmlp_v5',
                                   '042_realmlp_v5_seed2026_foldvariant',
                                   '044_shared001_updated_seed2027_foldvariant',
                                   '040_gpu_logreg_add039',
                                   '049_gpu_logreg_add048',
                                   '050_blend_add047_048_049_best'],
 'Z06_050_positive_core_no_051': ['015_realmlp_seed42',
                                  '026_realmlp_v5',
                                  '027_blend_add026',
                                  '028_cat_v3',
                                  '029_blend_add028',
                                  '031_blend_add030',
                                  '032_ovr_tabm',
                                  '033_blend_add032',
                                  '035_shared001_updated',
                                  '036_blend_add035',
                                  '039_cat_v3_seed2026_qbin_variant',
                                  '040_gpu_logreg_add039',
                                  '042_realmlp_v5_seed2026_foldvariant',
                                  '044_shared001_updated_seed2027_foldvariant',
                                  '045_gpu_logreg_add044',
                                  '049_gpu_logreg_add048'],
 'Z07_050_positive_core_add051': ['015_realmlp_seed42',
                                  '026_realmlp_v5',
                                  '027_blend_add026',
                                  '028_cat_v3',
                                  '029_blend_add028',
                                  '031_blend_add030',
                                  '032_ovr_tabm',
                                  '033_blend_add032',
                                  '035_shared001_updated',
                                  '036_blend_add035',
                                  '039_cat_v3_seed2026_qbin_variant',
                                  '040_gpu_logreg_add039',
                                  '042_realmlp_v5_seed2026_foldvariant',
                                  '044_shared001_updated_seed2027_foldvariant',
                                  '045_gpu_logreg_add044',
                                  '049_gpu_logreg_add048',
                                  '051_shared001_updated_v2'],
 'Z08_trimmed_full_with_050_051': ['015_realmlp_seed42',
                                   '026_realmlp_v5',
                                   '027_blend_add026',
                                   '028_cat_v3',
                                   '029_blend_add028',
                                   '031_blend_add030',
                                   '032_ovr_tabm',
                                   '033_blend_add032',
                                   '035_shared001_updated',
                                   '036_blend_add035',
                                   '038_gpu_logreg_add037',
                                   '039_cat_v3_seed2026_qbin_variant',
                                   '040_gpu_logreg_add039',
                                   '042_realmlp_v5_seed2026_foldvariant',
                                   '044_shared001_updated_seed2027_foldvariant',
                                   '045_gpu_logreg_add044',
                                   '049_gpu_logreg_add048',
                                   '050_blend_add047_048_049_best',
                                   '051_shared001_updated_v2']}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("NPY_DIR_FALLBACK:", NPY_DIR_FALLBACK)
print("RUN_IN_SAMPLE_META:", RUN_IN_SAMPLE_META)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))
print("omitted_models:", [x["key"] for x in OMITTED_MODEL_NOTES])


EXP_ID: exp_20260615_052_blend_add051_trimmed_check
OUTDIR: /kaggle/working/exp_20260615_052_blend_add051_trimmed_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
NPY_DIR_FALLBACK: /kaggle/input/datasets/mizushimatoshihiko/npy-files
RUN_IN_SAMPLE_META: False
n_models: 19
n_blend_sets: 33
omitted_models: ['017_realmlp_bias', '018_xgb_shared003', '019_blend_best', '020_blend_bias', '024_seed_ensemble_blend', '030_ovr_xgb', '034_gpu_logreg_default', '036_gpu_logreg_add035', '037_tabicl_v2', '047_xgb_multiclass_fe_objective_foldvariant', '048_ovr_xgb_seed2027_fe_foldvariant']


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then fallback dir, then scan /kaggle/input."""
    for base in [NPY_DIR, NPY_DIR_FALLBACK]:
        p = base / filename
        if p.exists():
            return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
loaded 029_blend_add028: (577347, 3) / (247435, 3)
loaded 031_blend_add030: (577347, 3) / (247435, 3)
loaded 032_ovr_tabm: (577347, 3) / (247435, 3)
loaded 033_blend_add032: (577347, 3) / (247435, 3)
loaded 035_shared001_updated: (577347, 3) / (247435, 3)
loaded 036_blend_add035: (577347, 3) / (247435, 3)
loaded 038_gpu_logreg_add037: (577347, 3) / (247435, 3)
loaded 039_cat_v3_seed2026_qbin_variant: (577347, 3) / (247435, 3)
loaded 040_gpu_logreg_add039: (577347, 3) / (247435, 3)
loaded 042_realmlp_v5_seed2026_foldvariant: (577347, 3) / (247435, 3)
loaded 044_shared001_updated_seed2027_foldvariant: (577347, 3) / (247435, 3)
loaded 045_gpu_logreg_add044: (577347, 3) / (247435, 3)
loaded 049_gpu_logreg_add048: (577347, 3) / (247435, 3)
loaded 050_blend_add047_048_049_best: (577347, 3) / (2474

In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "public_lb_biased": spec.get("public_lb_biased", np.nan),
        "tuned_cv": spec.get("tuned_cv", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus tables for newly added / current reference models.
def focus_pairwise(key, filename):
    df = pairwise_df[(pairwise_df["left"] == key) | (pairwise_df["right"] == key)].copy()
    df = df.sort_values("pearson_flat_proba").reset_index(drop=True)
    display(df)
    df.to_csv(OUTDIR / filename, index=False)
    return df

focus_pairwise_tables = {}
for short_name, key in FOCUS_MODEL_KEYS.items():
    if key in model_keys:
        focus_pairwise_tables[short_name] = focus_pairwise(key, f"pairwise_oof_correlation_focus_{short_name}.csv")

# Backward-compatible variables for easy notebook inspection.
focus_040_df = focus_pairwise_tables.get("040", pd.DataFrame())
focus_039_df = focus_pairwise_tables.get("039", pd.DataFrame())
focus_038_df = focus_pairwise_tables.get("038", pd.DataFrame())
focus_037_df = focus_pairwise_tables.get("037", pd.DataFrame())
focus_033_df = focus_pairwise_tables.get("033", pd.DataFrame())


,key,exp_id,family,role,cv_from_memo,public_lb,public_lb_biased,tuned_cv,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,050_blend_add047_048_049_best,exp_20260614_050_blend_add047_048_049_check,Blend,current_cv_best_blend_reference,0.970277,0.970520,NaN,NaN,0.970277,0.0,0.961320,0.976601,0.972910
1,045_gpu_logreg_add044,exp_20260612_045_gpu_logreg_stacker_add044_own,GPU_LogisticRegression_Stacker,cv_strong_stacker_positive_weight_in_050,0.970245,0.970400,NaN,0.970245,0.970245,0.0,0.960096,0.977148,0.973490
2,040_gpu_logreg_add039,exp_20260611_040_gpu_logreg_stacker_add039_own,GPU_LogisticRegression_Stacker,current_best_public_confirmed_positive_weight_...,0.970202,0.970690,NaN,NaN,0.970202,0.0,0.959500,0.976866,0.974240
3,049_gpu_logreg_add048,exp_20260614_049_gpu_logreg_stacker_add048_own,GPU_LogisticRegression_Stacker,largest_positive_weight_in_050,0.970196,0.970460,NaN,0.970196,0.970196,0.0,0.960273,0.977199,0.973115
4,038_gpu_logreg_add037,exp_20260610_038_gpu_logreg_stacker_add037_own,GPU_LogisticRegression_Stacker,public_lb_backup_keep_even_zero_weight_in_full...,0.970135,0.970600,NaN,NaN,0.970135,0.0,0.958120,0.977950,0.974336
5,036_blend_add035,exp_20260610_036_blend_add035_shared001_check,Blend,strong_positive_weight_in_050,0.970073,0.970230,NaN,NaN,0.970073,0.0,0.960777,0.976943,0.972499
6,033_blend_add032,exp_20260609_033_blend_add032_tabm_check,Blend,private_diversity_hedge_positive_weight_in_050,0.970040,0.970430,NaN,NaN,0.970040,0.0,0.961775,0.975773,0.972571
7,031_blend_add030,exp_20260608_031_blend_add030_ovr_xgb_check,Blend,public_confirmed_before_033_positive_weight_in...,0.970033,0.970430,NaN,NaN,0.970033,0.0,0.961738,0.975790,0.972571
8,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,previous_best_backup_positive_weight_in_050,0.970002,0.970036,NaN,NaN,0.970002,0.0,0.961315,0.975867,0.972825
9,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,previous_cv_trusted_slot_positive_weight_in_050,0.969543,0.969750,NaN,NaN,0.969543,0.0,0.961479,0.976439,0.970710


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,042_realmlp_v5_seed2026_foldvariant,0.987614,0.759791,0.022775,0.029062,0.035949,0.021237,0.485142,0.007825,0.014712
1,026_realmlp_v5,032_ovr_tabm,0.988073,0.761653,0.022150,0.035388,0.029062,0.021273,0.492699,0.014115,0.007789
2,032_ovr_tabm,039_cat_v3_seed2026_qbin_variant,0.988993,0.962610,0.022210,0.029062,0.035587,0.021362,0.493478,0.007701,0.014225
3,028_cat_v3,032_ovr_tabm,0.989126,0.963762,0.021803,0.035277,0.029062,0.021407,0.498608,0.013870,0.007656
4,032_ovr_tabm,044_shared001_updated_seed2027_foldvariant,0.989198,0.883758,0.022169,0.029062,0.035833,0.021491,0.495151,0.007571,0.014341
...,...,...,...,...,...,...,...,...,...,...,...
166,045_gpu_logreg_add044,049_gpu_logreg_add048,0.999960,0.998869,0.001136,0.034525,0.034452,0.033936,0.968464,0.000589,0.000516
167,040_gpu_logreg_add039,045_gpu_logreg_add044,0.999966,0.999174,0.001102,0.034865,0.034525,0.034158,0.969520,0.000707,0.000367
168,029_blend_add028,033_blend_add032,0.999993,0.999794,0.000438,0.034083,0.033838,0.033749,0.987632,0.000334,0.000088
169,029_blend_add028,031_blend_add030,0.999993,0.999801,0.000421,0.034083,0.033858,0.033768,0.988140,0.000315,0.000090


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,051_shared001_updated_v2,0.989217,0.868543,0.022016,0.029062,0.035597,0.021450,0.496412,0.007612,0.014147
1,039_cat_v3_seed2026_qbin_variant,051_shared001_updated_v2,0.995355,0.876749,0.013181,0.035587,0.035597,0.029133,0.692808,0.006454,0.006464
2,028_cat_v3,051_shared001_updated_v2,0.995524,0.870554,0.012951,0.035277,0.035597,0.029095,0.696406,0.006182,0.006502
3,042_realmlp_v5_seed2026_foldvariant,051_shared001_updated_v2,0.997292,0.808621,0.009178,0.035949,0.035597,0.031283,0.776951,0.004666,0.004315
4,015_realmlp_seed42,051_shared001_updated_v2,0.997476,0.949194,0.009388,0.034388,0.035597,0.030403,0.768083,0.003985,0.005194
5,044_shared001_updated_seed2027_foldvariant,051_shared001_updated_v2,0.997880,0.854347,0.008181,0.035833,0.035597,0.031716,0.798596,0.004117,0.003882
6,026_realmlp_v5,051_shared001_updated_v2,0.997924,0.819435,0.007919,0.035388,0.035597,0.031633,0.803829,0.003755,0.003965
7,038_gpu_logreg_add037,051_shared001_updated_v2,0.998042,0.881959,0.008371,0.035533,0.035597,0.031475,0.793710,0.004058,0.004122
8,045_gpu_logreg_add044,051_shared001_updated_v2,0.998050,0.883157,0.008265,0.034525,0.035597,0.031021,0.793355,0.003504,0.004576
9,040_gpu_logreg_add039,051_shared001_updated_v2,0.998058,0.885607,0.008220,0.034865,0.035597,0.031215,0.795357,0.003649,0.004382


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,050_blend_add047_048_049_best,0.991061,0.875170,0.020310,0.029062,0.033919,0.021434,0.515904,0.007628,0.012485
1,015_realmlp_seed42,050_blend_add047_048_049_best,0.997418,0.898910,0.009480,0.034388,0.033919,0.029520,0.761052,0.004869,0.004399
2,042_realmlp_v5_seed2026_foldvariant,050_blend_add047_048_049_best,0.998012,0.931579,0.007848,0.035949,0.033919,0.031089,0.801688,0.004860,0.002830
3,044_shared001_updated_seed2027_foldvariant,050_blend_add047_048_049_best,0.998165,0.883722,0.007884,0.035833,0.033919,0.031019,0.800868,0.004813,0.002899
4,039_cat_v3_seed2026_qbin_variant,050_blend_add047_048_049_best,0.998213,0.910029,0.008177,0.035587,0.033919,0.030746,0.793234,0.004841,0.003173
5,028_cat_v3,050_blend_add047_048_049_best,0.998330,0.910789,0.007987,0.035277,0.033919,0.030685,0.796798,0.004592,0.003234
6,035_shared001_updated,050_blend_add047_048_049_best,0.998340,0.874614,0.007384,0.034984,0.033919,0.030845,0.810449,0.004140,0.003074
7,026_realmlp_v5,050_blend_add047_048_049_best,0.998407,0.949607,0.007127,0.035388,0.033919,0.031172,0.817414,0.004216,0.002747
8,050_blend_add047_048_049_best,051_shared001_updated_v2,0.998434,0.878380,0.007406,0.033919,0.035597,0.031134,0.811146,0.002785,0.004464
9,027_blend_add026,050_blend_add047_048_049_best,0.999213,0.984369,0.005454,0.034163,0.033919,0.031375,0.854716,0.002789,0.002544


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,049_gpu_logreg_add048,0.990478,0.955091,0.021084,0.029062,0.034452,0.021320,0.505275,0.007742,0.013132
1,015_realmlp_seed42,049_gpu_logreg_add048,0.996940,0.927431,0.010295,0.034388,0.034452,0.029391,0.745039,0.004997,0.005061
2,042_realmlp_v5_seed2026_foldvariant,049_gpu_logreg_add048,0.997536,0.853026,0.008517,0.035949,0.034452,0.031042,0.788682,0.004907,0.003410
3,044_shared001_updated_seed2027_foldvariant,049_gpu_logreg_add048,0.997714,0.893090,0.008679,0.035833,0.034452,0.030896,0.784398,0.004936,0.003556
4,026_realmlp_v5,049_gpu_logreg_add048,0.997809,0.854673,0.008272,0.035388,0.034452,0.030883,0.792726,0.004505,0.003570
5,035_shared001_updated,049_gpu_logreg_add048,0.997836,0.903812,0.008439,0.034984,0.034452,0.030595,0.787692,0.004389,0.003857
6,039_cat_v3_seed2026_qbin_variant,049_gpu_logreg_add048,0.998055,0.971637,0.008279,0.035587,0.034452,0.030968,0.792579,0.004619,0.003485
7,049_gpu_logreg_add048,051_shared001_updated_v2,0.998073,0.884433,0.008142,0.034452,0.035597,0.031045,0.795950,0.003407,0.004552
8,028_cat_v3,049_gpu_logreg_add048,0.998184,0.973372,0.008089,0.035277,0.034452,0.030907,0.796110,0.004370,0.003546
9,027_blend_add026,049_gpu_logreg_add048,0.998850,0.915036,0.006391,0.034163,0.034452,0.031189,0.833349,0.002974,0.003263


...

,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,045_gpu_logreg_add044,0.990390,0.951425,0.021247,0.029062,0.034525,0.021271,0.502681,0.007791,0.013254
1,015_realmlp_seed42,045_gpu_logreg_add044,0.996933,0.926308,0.010306,0.034388,0.034525,0.029423,0.745044,0.004966,0.005103
2,042_realmlp_v5_seed2026_foldvariant,045_gpu_logreg_add044,0.997539,0.857610,0.008610,0.035949,0.034525,0.031032,0.786756,0.004917,0.003494
3,044_shared001_updated_seed2027_foldvariant,045_gpu_logreg_add044,0.997719,0.892361,0.008721,0.035833,0.034525,0.030914,0.783735,0.004919,0.003611
4,026_realmlp_v5,045_gpu_logreg_add044,0.997793,0.860130,0.008349,0.035388,0.034525,0.030884,0.791328,0.004503,0.003641
5,035_shared001_updated,045_gpu_logreg_add044,0.997819,0.901692,0.008534,0.034984,0.034525,0.030585,0.785743,0.004399,0.003940
6,045_gpu_logreg_add044,051_shared001_updated_v2,0.998050,0.883157,0.008265,0.034525,0.035597,0.031021,0.793355,0.003504,0.004576
7,039_cat_v3_seed2026_qbin_variant,045_gpu_logreg_add044,0.998123,0.969399,0.008170,0.035587,0.034525,0.031054,0.795078,0.004533,0.003471
8,028_cat_v3,045_gpu_logreg_add044,0.998216,0.970972,0.008068,0.035277,0.034525,0.030952,0.796701,0.004325,0.003573
9,027_blend_add026,045_gpu_logreg_add044,0.998862,0.918929,0.006377,0.034163,0.034525,0.031231,0.833765,0.002932,0.003294


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,036_blend_add035,0.990711,0.879875,0.020667,0.029062,0.034264,0.021429,0.511472,0.007633,0.012835
1,015_realmlp_seed42,036_blend_add035,0.997365,0.899824,0.009616,0.034388,0.034264,0.029623,0.759020,0.004765,0.004640
2,036_blend_add035,039_cat_v3_seed2026_qbin_variant,0.997845,0.912962,0.008929,0.034264,0.035587,0.030550,0.777347,0.003714,0.005037
3,036_blend_add035,042_realmlp_v5_seed2026_foldvariant,0.997856,0.923022,0.008200,0.034264,0.035949,0.031092,0.794784,0.003171,0.004857
4,036_blend_add035,044_shared001_updated_seed2027_foldvariant,0.998039,0.879508,0.008146,0.034264,0.035833,0.031065,0.795873,0.003199,0.004768
5,035_shared001_updated,036_blend_add035,0.998337,0.880209,0.007484,0.034984,0.034264,0.030968,0.808968,0.004017,0.003296
6,026_realmlp_v5,036_blend_add035,0.998402,0.946615,0.007200,0.035388,0.034264,0.031309,0.816551,0.004079,0.002955
7,036_blend_add035,051_shared001_updated_v2,0.998419,0.879960,0.007498,0.034264,0.035597,0.031262,0.809917,0.003002,0.004335
8,028_cat_v3,036_blend_add035,0.998477,0.915623,0.007612,0.035277,0.034264,0.031045,0.806479,0.004231,0.003218
9,027_blend_add026,036_blend_add035,0.999188,0.982346,0.005588,0.034163,0.034264,0.031478,0.851959,0.002685,0.002785


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,039_cat_v3_seed2026_qbin_variant,0.988993,0.962610,0.022210,0.029062,0.035587,0.021362,0.493478,0.007701,0.014225
1,015_realmlp_seed42,039_cat_v3_seed2026_qbin_variant,0.993330,0.916658,0.015687,0.034388,0.035587,0.027315,0.640276,0.007074,0.008272
2,026_realmlp_v5,039_cat_v3_seed2026_qbin_variant,0.994581,0.804236,0.014011,0.035388,0.035587,0.028631,0.676157,0.006757,0.006956
3,035_shared001_updated,039_cat_v3_seed2026_qbin_variant,0.994848,0.898905,0.013708,0.034984,0.035587,0.028567,0.680096,0.006417,0.007020
4,039_cat_v3_seed2026_qbin_variant,042_realmlp_v5_seed2026_foldvariant,0.995016,0.804879,0.013191,0.035587,0.035949,0.029310,0.694122,0.006277,0.006639
5,039_cat_v3_seed2026_qbin_variant,044_shared001_updated_seed2027_foldvariant,0.995318,0.892239,0.012985,0.035587,0.035833,0.029345,0.697431,0.006242,0.006488
6,039_cat_v3_seed2026_qbin_variant,051_shared001_updated_v2,0.995355,0.876749,0.013181,0.035587,0.035597,0.029133,0.692808,0.006454,0.006464
7,027_blend_add026,039_cat_v3_seed2026_qbin_variant,0.995823,0.871286,0.012699,0.034163,0.035587,0.028655,0.697294,0.005508,0.006932
8,038_gpu_logreg_add037,039_cat_v3_seed2026_qbin_variant,0.997587,0.963981,0.009395,0.035533,0.035587,0.030954,0.770634,0.004580,0.004633
9,036_blend_add035,039_cat_v3_seed2026_qbin_variant,0.997845,0.912962,0.008929,0.034264,0.035587,0.030550,0.777347,0.003714,0.005037


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,032_ovr_tabm,035_shared001_updated,0.989571,0.890900,0.021147,0.029062,0.034984,0.021580,0.508157,0.007483,0.013404
1,035_shared001_updated,039_cat_v3_seed2026_qbin_variant,0.994848,0.898905,0.013708,0.034984,0.035587,0.028567,0.680096,0.006417,0.007020
2,028_cat_v3,035_shared001_updated,0.995089,0.900310,0.013491,0.035277,0.034984,0.028527,0.683544,0.006750,0.006457
3,035_shared001_updated,042_realmlp_v5_seed2026_foldvariant,0.997450,0.816058,0.008688,0.034984,0.035949,0.031217,0.786001,0.003767,0.004732
4,035_shared001_updated,038_gpu_logreg_add037,0.997770,0.899700,0.008749,0.034984,0.035533,0.030988,0.783937,0.003996,0.004545
5,035_shared001_updated,040_gpu_logreg_add039,0.997811,0.902374,0.008593,0.034984,0.034865,0.030728,0.785487,0.004256,0.004136
6,035_shared001_updated,045_gpu_logreg_add044,0.997819,0.901692,0.008534,0.034984,0.034525,0.030585,0.785743,0.004399,0.003940
7,035_shared001_updated,049_gpu_logreg_add048,0.997836,0.903812,0.008439,0.034984,0.034452,0.030595,0.787692,0.004389,0.003857
8,015_realmlp_seed42,035_shared001_updated,0.997978,0.893795,0.008044,0.034388,0.034984,0.030753,0.796295,0.003636,0.004231
9,035_shared001_updated,044_shared001_updated_seed2027_foldvariant,0.998062,0.870384,0.008078,0.034984,0.035833,0.031446,0.798689,0.003539,0.004387


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,015_realmlp_seed42,032_ovr_tabm,0.989870,0.909912,0.020289,0.034388,0.029062,0.021724,0.520609,0.012665,0.007339
1,015_realmlp_seed42,039_cat_v3_seed2026_qbin_variant,0.993330,0.916658,0.015687,0.034388,0.035587,0.027315,0.640276,0.007074,0.008272
2,015_realmlp_seed42,028_cat_v3,0.993619,0.913554,0.015279,0.034388,0.035277,0.027358,0.646647,0.007030,0.007919
3,015_realmlp_seed42,042_realmlp_v5_seed2026_foldvariant,0.996174,0.813865,0.011006,0.034388,0.035949,0.029778,0.734167,0.004611,0.006171
4,015_realmlp_seed42,044_shared001_updated_seed2027_foldvariant,0.996538,0.876838,0.010829,0.034388,0.035833,0.029802,0.737316,0.004586,0.006031
5,015_realmlp_seed42,038_gpu_logreg_add037,0.996779,0.922695,0.010654,0.034388,0.035533,0.029757,0.740869,0.004632,0.005776
6,015_realmlp_seed42,040_gpu_logreg_add039,0.996843,0.927797,0.010510,0.034388,0.034865,0.029495,0.741875,0.004893,0.005369
7,015_realmlp_seed42,045_gpu_logreg_add044,0.996933,0.926308,0.010306,0.034388,0.034525,0.029423,0.745044,0.004966,0.005103
8,015_realmlp_seed42,049_gpu_logreg_add048,0.996940,0.927431,0.010295,0.034388,0.034452,0.029391,0.745039,0.004997,0.005061
9,015_realmlp_seed42,031_blend_add030,0.997199,0.868061,0.009734,0.034388,0.033858,0.029364,0.755178,0.005025,0.004495


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def _weight_of(keys, weights, target_key):
    if weights is None or target_key not in keys:
        return None
    return float(dict(zip(keys, weights)).get(target_key, 0.0))

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    for short_name, target_key in TRACK_KEYS.items():
        row[f"includes_{short_name}"] = target_key in keys
        row[f"weight_{short_name}"] = _weight_of(keys, weights, target_key)
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models.
    # Diagnostic only; disabled by default in 052 to reduce runtime.
    if RUN_IN_SAMPLE_META:
        for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
            X_meta = build_meta_features(keys, oof, mode)
            X_test_meta = build_meta_features(keys, pred, mode)
            try:
                lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
                lr.fit(X_meta, y)
                record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
            except Exception as e:
                print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
            try:
                rc = RidgeClassifier(alpha=10.0, random_state=SEED)
                rc.fit(X_meta, y)
                record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
            except Exception as e:
                print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
            try:
                y_oh = onehot_y(y, n_classes)
                rr = Ridge(alpha=10.0, random_state=SEED)
                rr.fit(X_meta, y_oh)
                record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
            except Exception as e:
                print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
            try:
                y_oh = onehot_y(y, n_classes)
                oof_cols, test_cols = [], []
                for c in range(n_classes):
                    en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                    en.fit(X_meta, y_oh[:, c])
                    oof_cols.append(en.predict(X_meta))
                    test_cols.append(en.predict(X_test_meta))
                en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
                en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
                record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
            except Exception as e:
                print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(100))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_blend_tables = {}
for short_name, target_key in TRACK_KEYS.items():
    include_col = f"includes_{short_name}"
    if include_col in safe_blend_df.columns:
        df = safe_blend_df[safe_blend_df[include_col]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
        focus_blend_tables[short_name] = df
        display(df.head(80))
        df.to_csv(OUTDIR / f"safe_blend_diagnostics_focus_{short_name}.csv", index=False)

# Backward-compatible variables for easy notebook inspection.
focus_051_blend_df = focus_blend_tables.get("051", pd.DataFrame())
focus_050_blend_df = focus_blend_tables.get("050", pd.DataFrame())
focus_049_blend_df = focus_blend_tables.get("049", pd.DataFrame())
focus_040_blend_df = focus_blend_tables.get("040", pd.DataFrame())
focus_039_blend_df = focus_blend_tables.get("039", pd.DataFrame())
focus_038_blend_df = focus_blend_tables.get("038", pd.DataFrame())
focus_037_blend_df = focus_blend_tables.get("037", pd.DataFrame())
focus_033_blend_df = focus_blend_tables.get("033", pd.DataFrame())



=== A_051_only ['051_shared001_updated_v2'] ===

=== B_050_only ['050_blend_add047_048_049_best'] ===

=== C_049_only ['049_gpu_logreg_add048'] ===

=== D_040_only ['040_gpu_logreg_add039'] ===

=== E_038_only ['038_gpu_logreg_add037'] ===

=== F_033_only ['033_blend_add032'] ===

=== G_045_only ['045_gpu_logreg_add044'] ===

=== H_036_blend_only ['036_blend_add035'] ===

=== I_039_only ['039_cat_v3_seed2026_qbin_variant'] ===

=== J_035_only ['035_shared001_updated'] ===

=== K_015_only ['015_realmlp_seed42'] ===

=== L_051_040 ['051_shared001_updated_v2', '040_gpu_logreg_add039'] ===

=== M_051_038 ['051_shared001_updated_v2', '038_gpu_logreg_add037'] ===

=== N_051_033 ['051_shared001_updated_v2', '033_blend_add032'] ===

=== O_051_045 ['051_shared001_updated_v2', '045_gpu_logreg_add044'] ===

=== P_051_049 ['051_shared001_updated_v2', '049_gpu_logreg_add048'] ===

=== Q_051_050 ['051_shared001_updated_v2', '050_blend_add047_048_049_best'] ===

=== R_051_036blend ['051_shared001_up

,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z05_051_plus_realmlp_and_core,hc_nonnegative_raw,"051_shared001_updated_v2,026_realmlp_v5,042_re...",7,0.970281,"{""051_shared001_updated_v2"": 0.0, ""026_realmlp...",True,0.000000e+00,True,0.038416,...,False,NaN,0.960239,0.977139,0.973466,0.970281,17.0,NaN,NaN,NaN
1,B_050_only,hc_nonnegative_raw,050_blend_add047_048_049_best,1,0.970277,"{""050_blend_add047_048_049_best"": 1.0}",False,NaN,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,1.0,NaN,NaN,NaN
2,B_050_only,nm_softmax_raw,050_blend_add047_048_049_best,1,0.970277,"{""050_blend_add047_048_049_best"": 1.0}",False,NaN,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
3,Q_051_050,nm_softmax_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 2.379031740900678...",True,2.379032e-07,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
4,Q_051_050,hc_nonnegative_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 0.0, ""050_blend_a...",True,0.000000e+00,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,8.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,I_039_only,avg,039_cat_v3_seed2026_qbin_variant,1,0.968705,None,False,NaN,False,NaN,...,False,NaN,0.959571,0.974928,0.971616,NaN,NaN,NaN,NaN,NaN
95,I_039_only,nm_softmax_raw,039_cat_v3_seed2026_qbin_variant,1,0.968705,"{""039_cat_v3_seed2026_qbin_variant"": 1.0}",False,NaN,False,NaN,...,False,NaN,0.959571,0.974928,0.971616,NaN,NaN,0.968705,True,Optimization terminated successfully.
96,K_015_only,hc_nonnegative_raw,015_realmlp_seed42,1,0.968169,"{""015_realmlp_seed42"": 1.0}",False,NaN,False,NaN,...,True,1.0,0.962234,0.976098,0.966177,0.968169,1.0,NaN,NaN,NaN
97,K_015_only,avg,015_realmlp_seed42,1,0.968169,None,False,NaN,False,NaN,...,True,NaN,0.962234,0.976098,0.966177,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z05_051_plus_realmlp_and_core,hc_nonnegative_raw,"051_shared001_updated_v2,026_realmlp_v5,042_re...",7,0.970281,"{""051_shared001_updated_v2"": 0.0, ""026_realmlp...",True,0.000000e+00,True,0.038416,...,False,NaN,0.960239,0.977139,0.973466,0.970281,17.0,NaN,NaN,NaN
1,B_050_only,hc_nonnegative_raw,050_blend_add047_048_049_best,1,0.970277,"{""050_blend_add047_048_049_best"": 1.0}",False,NaN,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,1.0,NaN,NaN,NaN
2,B_050_only,nm_softmax_raw,050_blend_add047_048_049_best,1,0.970277,"{""050_blend_add047_048_049_best"": 1.0}",False,NaN,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
3,Q_051_050,nm_softmax_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 2.379031740900678...",True,2.379032e-07,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
4,Q_051_050,hc_nonnegative_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 0.0, ""050_blend_a...",True,0.000000e+00,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,8.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,I_039_only,avg,039_cat_v3_seed2026_qbin_variant,1,0.968705,None,False,NaN,False,NaN,...,False,NaN,0.959571,0.974928,0.971616,NaN,NaN,NaN,NaN,NaN
95,I_039_only,nm_softmax_raw,039_cat_v3_seed2026_qbin_variant,1,0.968705,"{""039_cat_v3_seed2026_qbin_variant"": 1.0}",False,NaN,False,NaN,...,False,NaN,0.959571,0.974928,0.971616,NaN,NaN,0.968705,True,Optimization terminated successfully.
96,K_015_only,hc_nonnegative_raw,015_realmlp_seed42,1,0.968169,"{""015_realmlp_seed42"": 1.0}",False,NaN,False,NaN,...,True,1.0,0.962234,0.976098,0.966177,0.968169,1.0,NaN,NaN,NaN
97,K_015_only,avg,015_realmlp_seed42,1,0.968169,None,False,NaN,False,NaN,...,True,NaN,0.962234,0.976098,0.966177,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z05_051_plus_realmlp_and_core,hc_nonnegative_raw,"051_shared001_updated_v2,026_realmlp_v5,042_re...",7,0.970281,"{""051_shared001_updated_v2"": 0.0, ""026_realmlp...",True,0.000000e+00,True,0.038416,...,False,NaN,0.960239,0.977139,0.973466,0.970281,17.0,NaN,NaN,NaN
1,Q_051_050,nm_softmax_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 2.379031740900678...",True,2.379032e-07,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
2,Q_051_050,hc_nonnegative_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 0.0, ""050_blend_a...",True,0.000000e+00,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,8.0,NaN,NaN,NaN
3,V_051_045_040_038,hc_nonnegative_raw,"051_shared001_updated_v2,045_gpu_logreg_add044...",4,0.970255,"{""051_shared001_updated_v2"": 0.009852221602079...",True,9.852222e-03,False,NaN,...,False,NaN,0.959590,0.977250,0.973925,0.970255,13.0,NaN,NaN,NaN
4,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,3.969607e-02,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Z03_shared001_lineage_015_035_051,hc_nonnegative_raw,"015_realmlp_seed42,035_shared001_updated,051_s...",3,0.968992,"{""015_realmlp_seed42"": 0.33252268806523677, ""0...",True,3.647127e-01,False,NaN,...,True,0.332523,0.961254,0.976234,0.969489,0.968992,5.0,NaN,NaN,NaN
62,Z03_shared001_lineage_015_035_051,avg,"015_realmlp_seed42,035_shared001_updated,051_s...",3,0.968967,None,True,NaN,False,NaN,...,True,NaN,0.961288,0.976234,0.969380,NaN,NaN,NaN,NaN,NaN
63,A_051_only,hc_nonnegative_raw,051_shared001_updated_v2,1,0.968908,"{""051_shared001_updated_v2"": 1.0}",True,1.000000e+00,False,NaN,...,False,NaN,0.959264,0.975927,0.971532,0.968908,1.0,NaN,NaN,NaN
64,A_051_only,avg,051_shared001_updated_v2,1,0.968908,None,True,NaN,False,NaN,...,False,NaN,0.959264,0.975927,0.971532,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z05_051_plus_realmlp_and_core,hc_nonnegative_raw,"051_shared001_updated_v2,026_realmlp_v5,042_re...",7,0.970281,"{""051_shared001_updated_v2"": 0.0, ""026_realmlp...",True,0.000000e+00,True,0.038416,...,False,NaN,0.960239,0.977139,0.973466,0.970281,17.0,NaN,NaN,NaN
1,B_050_only,hc_nonnegative_raw,050_blend_add047_048_049_best,1,0.970277,"{""050_blend_add047_048_049_best"": 1.0}",False,NaN,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,1.0,NaN,NaN,NaN
2,B_050_only,nm_softmax_raw,050_blend_add047_048_049_best,1,0.970277,"{""050_blend_add047_048_049_best"": 1.0}",False,NaN,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
3,Q_051_050,nm_softmax_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 2.379031740900678...",True,2.379032e-07,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,0.970277,True,Optimization terminated successfully.
4,Q_051_050,hc_nonnegative_raw,"051_shared001_updated_v2,050_blend_add047_048_...",2,0.970277,"{""051_shared001_updated_v2"": 0.0, ""050_blend_a...",True,0.000000e+00,True,1.000000,...,False,NaN,0.961320,0.976601,0.972910,0.970277,8.0,NaN,NaN,NaN
5,B_050_only,avg,050_blend_add047_048_049_best,1,0.970277,None,False,NaN,True,NaN,...,False,NaN,0.961320,0.976601,0.972910,NaN,NaN,NaN,NaN,NaN
6,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,3.969607e-02,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
7,Z02_051_050_049_045_040_038_033,hc_nonnegative_raw,"051_shared001_updated_v2,050_blend_add047_048_...",7,0.970241,"{""051_shared001_updated_v2"": 0.173865359434349...",True,1.738654e-01,True,0.220692,...,False,NaN,0.960567,0.976763,0.973393,0.970241,8.0,NaN,NaN,NaN
8,Z01_051_050_049_040_038,hc_nonnegative_raw,"051_shared001_updated_v2,050_blend_add047_048_...",5,0.970192,"{""051_shared001_updated_v2"": 0.208283063971326...",True,2.082831e-01,True,0.171893,...,False,NaN,0.959900,0.977088,0.973587,0.970192,9.0,NaN,NaN,NaN
9,Z01_051_050_049_040_038,nm_softmax_raw,"051_shared001_updated_v2,050_blend_add047_048_...",5,0.970183,"{""051_shared001_updated_v2"": 0.165168027462883...",True,1.651680e-01,True,0.223348,...,False,NaN,0.959966,0.977020,0.973563,NaN,NaN,0.970183,True,Optimization terminated successfully.


...

,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z06_050_positive_core_no_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970257,"{""015_realmlp_seed42"": 0.0, ""026_realmlp_v5"": ...",False,NaN,False,NaN,...,True,0.000000,0.961648,0.976516,0.972608,0.970257,14.0,NaN,NaN,NaN
1,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,0.039696,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
2,Z07_050_positive_core_add051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970215,"{""015_realmlp_seed42"": 0.04610065720852583, ""0...",True,0.046101,False,NaN,...,True,0.046101,0.962006,0.976260,0.972378,0.970215,7.0,NaN,NaN,NaN
3,Z08_trimmed_full_with_050_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970096,"{""015_realmlp_seed42"": 0.04942105937045611, ""0...",True,0.040561,True,0.048636,...,True,0.049421,0.961985,0.976132,0.972173,NaN,NaN,0.970096,True,Optimization terminated successfully.
4,Z06_050_positive_core_no_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970057,"{""015_realmlp_seed42"": 0.07221283439046085, ""0...",False,NaN,False,NaN,...,True,0.072213,0.962334,0.976063,0.971774,NaN,NaN,0.970057,True,Optimization terminated successfully.
5,Z08_trimmed_full_with_050_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970022,None,True,NaN,True,NaN,...,True,NaN,0.962067,0.976140,0.971858,NaN,NaN,NaN,NaN,NaN
6,Z07_050_positive_core_add051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,None,True,NaN,False,NaN,...,True,NaN,0.962337,0.976012,0.971677,NaN,NaN,NaN,NaN,NaN
7,Z07_050_positive_core_add051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,"{""015_realmlp_seed42"": 0.058823529411764705, ""...",True,0.058824,False,NaN,...,True,0.058824,0.962337,0.976012,0.971677,NaN,NaN,0.970009,True,Optimization terminated successfully.
8,Z06_050_positive_core_no_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970003,None,False,NaN,False,NaN,...,True,NaN,0.962462,0.975978,0.971568,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z06_050_positive_core_no_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970257,"{""015_realmlp_seed42"": 0.0, ""026_realmlp_v5"": ...",False,NaN,False,NaN,...,True,0.000000,0.961648,0.976516,0.972608,0.970257,14.0,NaN,NaN,NaN
1,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,0.039696,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
2,Z07_050_positive_core_add051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970215,"{""015_realmlp_seed42"": 0.04610065720852583, ""0...",True,0.046101,False,NaN,...,True,0.046101,0.962006,0.976260,0.972378,0.970215,7.0,NaN,NaN,NaN
3,Z08_trimmed_full_with_050_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970096,"{""015_realmlp_seed42"": 0.04942105937045611, ""0...",True,0.040561,True,0.048636,...,True,0.049421,0.961985,0.976132,0.972173,NaN,NaN,0.970096,True,Optimization terminated successfully.
4,Z06_050_positive_core_no_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970057,"{""015_realmlp_seed42"": 0.07221283439046085, ""0...",False,NaN,False,NaN,...,True,0.072213,0.962334,0.976063,0.971774,NaN,NaN,0.970057,True,Optimization terminated successfully.
5,Z08_trimmed_full_with_050_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970022,None,True,NaN,True,NaN,...,True,NaN,0.962067,0.976140,0.971858,NaN,NaN,NaN,NaN,NaN
6,Z07_050_positive_core_add051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,None,True,NaN,False,NaN,...,True,NaN,0.962337,0.976012,0.971677,NaN,NaN,NaN,NaN,NaN
7,Z07_050_positive_core_add051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,"{""015_realmlp_seed42"": 0.058823529411764705, ""...",True,0.058824,False,NaN,...,True,0.058824,0.962337,0.976012,0.971677,NaN,NaN,0.970009,True,Optimization terminated successfully.
8,Z06_050_positive_core_no_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970003,None,False,NaN,False,NaN,...,True,NaN,0.962462,0.975978,0.971568,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z06_050_positive_core_no_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970257,"{""015_realmlp_seed42"": 0.0, ""026_realmlp_v5"": ...",False,NaN,False,NaN,...,True,0.000000,0.961648,0.976516,0.972608,0.970257,14.0,NaN,NaN,NaN
1,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,0.039696,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
2,Z07_050_positive_core_add051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970215,"{""015_realmlp_seed42"": 0.04610065720852583, ""0...",True,0.046101,False,NaN,...,True,0.046101,0.962006,0.976260,0.972378,0.970215,7.0,NaN,NaN,NaN
3,Z08_trimmed_full_with_050_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970096,"{""015_realmlp_seed42"": 0.04942105937045611, ""0...",True,0.040561,True,0.048636,...,True,0.049421,0.961985,0.976132,0.972173,NaN,NaN,0.970096,True,Optimization terminated successfully.
4,Z06_050_positive_core_no_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970057,"{""015_realmlp_seed42"": 0.07221283439046085, ""0...",False,NaN,False,NaN,...,True,0.072213,0.962334,0.976063,0.971774,NaN,NaN,0.970057,True,Optimization terminated successfully.
5,Z08_trimmed_full_with_050_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970022,None,True,NaN,True,NaN,...,True,NaN,0.962067,0.976140,0.971858,NaN,NaN,NaN,NaN,NaN
6,Z07_050_positive_core_add051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,None,True,NaN,False,NaN,...,True,NaN,0.962337,0.976012,0.971677,NaN,NaN,NaN,NaN,NaN
7,Z07_050_positive_core_add051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,"{""015_realmlp_seed42"": 0.058823529411764705, ""...",True,0.058824,False,NaN,...,True,0.058824,0.962337,0.976012,0.971677,NaN,NaN,0.970009,True,Optimization terminated successfully.
8,Z06_050_positive_core_no_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970003,None,False,NaN,False,NaN,...,True,NaN,0.962462,0.975978,0.971568,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z05_051_plus_realmlp_and_core,hc_nonnegative_raw,"051_shared001_updated_v2,026_realmlp_v5,042_re...",7,0.970281,"{""051_shared001_updated_v2"": 0.0, ""026_realmlp...",True,0.000000,True,0.038416,...,False,NaN,0.960239,0.977139,0.973466,0.970281,17.0,NaN,NaN,NaN
1,Z06_050_positive_core_no_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970257,"{""015_realmlp_seed42"": 0.0, ""026_realmlp_v5"": ...",False,NaN,False,NaN,...,True,0.000000,0.961648,0.976516,0.972608,0.970257,14.0,NaN,NaN,NaN
2,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,0.039696,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
3,Z07_050_positive_core_add051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970215,"{""015_realmlp_seed42"": 0.04610065720852583, ""0...",True,0.046101,False,NaN,...,True,0.046101,0.962006,0.976260,0.972378,0.970215,7.0,NaN,NaN,NaN
4,Z08_trimmed_full_with_050_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970096,"{""015_realmlp_seed42"": 0.04942105937045611, ""0...",True,0.040561,True,0.048636,...,True,0.049421,0.961985,0.976132,0.972173,NaN,NaN,0.970096,True,Optimization terminated successfully.
5,Z05_051_plus_realmlp_and_core,nm_softmax_raw,"051_shared001_updated_v2,026_realmlp_v5,042_re...",7,0.970058,"{""051_shared001_updated_v2"": 0.159700151915257...",True,0.159700,True,0.135979,...,False,NaN,0.960141,0.976832,0.973200,NaN,NaN,0.970058,True,Optimization terminated successfully.
6,Z06_050_positive_core_no_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970057,"{""015_realmlp_seed42"": 0.07221283439046085, ""0...",False,NaN,False,NaN,...,True,0.072213,0.962334,0.976063,0.971774,NaN,NaN,0.970057,True,Optimization terminated successfully.
7,Z08_trimmed_full_with_050_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970022,None,True,NaN,True,NaN,...,True,NaN,0.962067,0.976140,0.971858,NaN,NaN,NaN,NaN,NaN
8,Z07_050_positive_core_add051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,None,True,NaN,False,NaN,...,True,NaN,0.962337,0.976012,0.971677,NaN,NaN,NaN,NaN,NaN
9,Z07_050_positive_core_add051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,"{""015_realmlp_seed42"": 0.058823529411764705, ""...",True,0.058824,False,NaN,...,True,0.058824,0.962337,0.976012,0.971677,NaN,NaN,0.970009,True,Optimization terminated successfully.


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_051,weight_051,includes_050,weight_050,...,includes_015,weight_015,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message
0,Z06_050_positive_core_no_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970257,"{""015_realmlp_seed42"": 0.0, ""026_realmlp_v5"": ...",False,NaN,False,NaN,...,True,0.000000,0.961648,0.976516,0.972608,0.970257,14.0,NaN,NaN,NaN
1,Z08_trimmed_full_with_050_051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970250,"{""015_realmlp_seed42"": 0.03969607195532618, ""0...",True,0.039696,True,0.004975,...,True,0.039696,0.961447,0.976610,0.972692,0.970250,10.0,NaN,NaN,NaN
2,Z07_050_positive_core_add051,hc_nonnegative_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970215,"{""015_realmlp_seed42"": 0.04610065720852583, ""0...",True,0.046101,False,NaN,...,True,0.046101,0.962006,0.976260,0.972378,0.970215,7.0,NaN,NaN,NaN
3,Z08_trimmed_full_with_050_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970096,"{""015_realmlp_seed42"": 0.04942105937045611, ""0...",True,0.040561,True,0.048636,...,True,0.049421,0.961985,0.976132,0.972173,NaN,NaN,0.970096,True,Optimization terminated successfully.
4,Z06_050_positive_core_no_051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970057,"{""015_realmlp_seed42"": 0.07221283439046085, ""0...",False,NaN,False,NaN,...,True,0.072213,0.962334,0.976063,0.971774,NaN,NaN,0.970057,True,Optimization terminated successfully.
5,Z08_trimmed_full_with_050_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",19,0.970022,None,True,NaN,True,NaN,...,True,NaN,0.962067,0.976140,0.971858,NaN,NaN,NaN,NaN,NaN
6,Z07_050_positive_core_add051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,None,True,NaN,False,NaN,...,True,NaN,0.962337,0.976012,0.971677,NaN,NaN,NaN,NaN,NaN
7,Z07_050_positive_core_add051,nm_softmax_raw,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",17,0.970009,"{""015_realmlp_seed42"": 0.058823529411764705, ""...",True,0.058824,False,NaN,...,True,0.058824,0.962337,0.976012,0.971677,NaN,NaN,0.970009,True,Optimization terminated successfully.
8,Z06_050_positive_core_no_051,avg,"015_realmlp_seed42,026_realmlp_v5,027_blend_ad...",16,0.970003,None,False,NaN,False,NaN,...,True,NaN,0.962462,0.975978,0.971568,NaN,NaN,NaN,NaN,NaN
9,Z03_shared001_lineage_015_035_051,nm_softmax_raw,"015_realmlp_seed42,035_shared001_updated,051_s...",3,0.969010,"{""015_realmlp_seed42"": 0.3325218185720948, ""03...",True,0.423265,False,NaN,...,True,0.332522,0.961171,0.976285,0.969574,NaN,NaN,0.969010,True,Optimization terminated successfully.


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / f"oof_{EXP_ID}_best_blend_proba.npy"
best_pred_npy = OUTDIR / f"pred_{EXP_ID}_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / f"submission_{EXP_ID}_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
for short_name, target_key in TRACK_KEYS.items():
    best_info[f"includes_{short_name}"] = target_key in best_keys
    best_info[f"weight_{short_name}"] = best_row.get(f"weight_{short_name}")

save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_record = {
    "set_name": best_set_name,
    "method": best_method,
    "cv": float(best_oof_score),
    "keys": ",".join(best_keys),
    "weights_json": best_row.get("weights_json"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}
for short_name, target_key in TRACK_KEYS.items():
    saved_submission_record[f"includes_{short_name}"] = target_key in best_keys
    saved_submission_record[f"weight_{short_name}"] = best_row.get(f"weight_{short_name}")

saved_submission_df = pd.DataFrame([saved_submission_record])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "Z05_051_plus_realmlp_and_core",
  "method": "hc_nonnegative_raw",
  "keys": "051_shared001_updated_v2,026_realmlp_v5,042_realmlp_v5_seed2026_foldvariant,044_shared001_updated_seed2027_foldvariant,040_gpu_logreg_add039,049_gpu_logreg_add048,050_blend_add047_048_049_best",
  "n_models": 7,
  "balanced_accuracy": 0.9702813295320772,
  "weights_json": "{\"051_shared001_updated_v2\": 0.0, \"026_realmlp_v5\": 0.08538166020688433, \"042_realmlp_v5_seed2026_foldvariant\": 0.08051551777352962, \"044_shared001_updated_seed2027_foldvariant\": 0.049030562247666026, \"040_gpu_logreg_add039\": 0.20627077091992446, \"049_gpu_logreg_add048\": 0.5403850688649761, \"050_blend_add047_048_049_best\": 0.03841641998701947}",
  "includes_051": true,
  "weight_051": 0.0,
  "includes_050": true,
  "weight_050": 0.03841641998701947,
  "includes_049": true,
  "weight_049": 0.5403850688649761,
  "includes_045": false,
  "weight_045": NaN,
  "includes_044": true,
  "weight_044": 0

,set_name,method,cv,keys,weights_json,submission,oof_proba,pred_proba,includes_051,weight_051,...,includes_031,weight_031,includes_029,weight_029,includes_028,weight_028,includes_026,weight_026,includes_015,weight_015
0,Z05_051_plus_realmlp_and_core,hc_nonnegative_raw,0.970281,"051_shared001_updated_v2,026_realmlp_v5,042_re...","{""051_shared001_updated_v2"": 0.0, ""026_realmlp...",submission_exp_20260615_052_blend_add051_trimm...,oof_exp_20260615_052_blend_add051_trimmed_chec...,pred_exp_20260615_052_blend_add051_trimmed_che...,True,0.0,...,False,NaN,False,NaN,False,NaN,True,0.085382,False,NaN


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "052_context": "trimmed blend/correlation check after adding 051 to 050 framework",
    "051_shared001_updated_v2_cv": 0.9689075888054314,
    "051_shared001_updated_v2_public_lb": 0.97055,
    "050_blend_add047_048_049_cv": 0.9702769901284838,
    "050_blend_add047_048_049_public_lb": 0.97052,
    "049_gpu_logreg_add048_cv": 0.9701958734042008,
    "049_gpu_logreg_add048_public_lb": 0.97046,
    "045_gpu_logreg_add044_cv": 0.9702445493383917,
    "045_gpu_logreg_add044_public_lb": 0.97040,
    "040_gpu_logreg_add039_cv": 0.9702017877238539,
    "040_gpu_logreg_add039_public_lb": 0.97069,
    "038_gpu_logreg_add037_cv": 0.9701353365798572,
    "038_gpu_logreg_add037_public_lb": 0.97060,
    "033_blend_add032_cv": 0.9700400336552478,
    "033_blend_add032_public_lb": 0.97043,
    "035_shared001_updated_cv": 0.9687410900866934,
    "035_shared001_updated_public_lb": 0.97012,
    "015_shared001_cv": 0.9681693449222352,
    "015_shared001_public_lb": 0.96977,
}

best_focus_safe = {}
for short_name, df in focus_blend_tables.items():
    best_focus_safe[short_name] = None if len(df) == 0 else df.iloc[0].to_dict()

judgement = {
    "reference_scores": reference_scores,
    "omitted_model_notes": OMITTED_MODEL_NOTES,
    "run_in_sample_meta": RUN_IN_SAMPLE_META,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_focus_safe": best_focus_safe,
    "main_questions": {
        "does_051_help_probability_blend": "Check focus_051 safe blends and weight_051 versus 050/040/038 references.",
        "does_051_replace_old_shared001_lineage": "Compare 051 with 015/035/044 in shared001 lineage sets.",
        "does_050_remain_useful_after_051": "Check focus_050 and combined 050+051 blend sets.",
        "should_promote_052_best_blend": "Only if best safe blend improves CV and later Public LB over current slot_1 040.",
    },
    "automatic_view": {
        "best_safe_includes_051": bool(best_info.get("includes_051")),
        "best_safe_weight_051": best_info.get("weight_051"),
        "best_safe_includes_050": bool(best_info.get("includes_050")),
        "best_safe_weight_050": best_info.get("weight_050"),
        "best_safe_includes_049": bool(best_info.get("includes_049")),
        "best_safe_weight_049": best_info.get("weight_049"),
        "best_safe_includes_040": bool(best_info.get("includes_040")),
        "best_safe_weight_040": best_info.get("weight_040"),
        "best_safe_includes_038": bool(best_info.get("includes_038")),
        "best_safe_weight_038": best_info.get("weight_038"),
        "best_safe_includes_033": bool(best_info.get("includes_033")),
        "best_safe_weight_033": best_info.get("weight_033"),
        "best_safe_cv": float(best_info["cv_score"]),
        "040_reference_cv": reference_scores["040_gpu_logreg_add039_cv"],
        "040_reference_public_lb": reference_scores["040_gpu_logreg_add039_public_lb"],
        "050_reference_cv": reference_scores["050_blend_add047_048_049_cv"],
        "050_reference_public_lb": reference_scores["050_blend_add047_048_049_public_lb"],
        "051_reference_cv": reference_scores["051_shared001_updated_v2_cv"],
        "051_reference_public_lb": reference_scores["051_shared001_updated_v2_public_lb"],
        "promotion_rule": (
            "Promote 052 only if the best safe probability blend improves over 040 and confirms on Public LB. "
            "Do not promote in-sample meta rows; RUN_IN_SAMPLE_META is disabled by default."
        ),
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Trimmed blend/correlation check after adding 051 to the 050 framework",
    "npy_dir": str(NPY_DIR),
    "npy_dir_fallback": str(NPY_DIR_FALLBACK),
    "run_in_sample_meta": RUN_IN_SAMPLE_META,
    "model_keys": model_keys,
    "omitted_model_notes": OMITTED_MODEL_NOTES,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus": {k: v.to_dict(orient="records") for k, v in focus_pairwise_tables.items()},
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus": {k: v.to_dict(orient="records") for k, v in focus_blend_tables.items()},
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
summary_path = OUTDIR / "blend_add051_trimmed_summary.json"
save_json(summary, summary_path)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Trimmed blend/correlation check after adding 051",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-15",
    },
    "objective": {
        "summary": (
            "Use the 050 blend framework, add 051 shared001 updated v2, and reduce runtime by omitting models "
            "that had zero/nearly zero weight in 050 or were superseded. Correlation and safe probability blend diagnostics remain enabled."
        ),
        "success_criteria": [
            "load trimmed OOF/pred npy pool including 051 and 050 best blend",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations with focus tables for 051/050/049/040/038/033",
            "evaluate targeted 051 blend sets",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {
        "npy_dir": str(NPY_DIR),
        "npy_dir_fallback": str(NPY_DIR_FALLBACK),
        "models": resolved_specs,
        "omitted_models": OMITTED_MODEL_NOTES,
        "blend_sets": BLEND_SETS,
        "run_in_sample_meta": RUN_IN_SAMPLE_META,
    },
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_prefix": "pairwise_oof_correlation_focus_*.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_prefix": "safe_blend_diagnostics_focus_*.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": summary_path.name,
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "051 has strong Public LB but only moderate CV. Use 052 to check whether it receives safe-blend weight "
            "and whether it improves the 050/040/038 cluster. 040 remains slot_1 unless 052 confirms on Public LB."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review safe_blend_diagnostics_focus_051.csv",
            "Check whether 051 receives non-trivial hc_nonnegative_raw or nm_softmax_raw weight",
            "Compare best safe blend against 050 CV 0.9702769901284838 / LB 0.97052 and 040 LB 0.97069",
            "Submit only the best safe blend, not optional in-sample meta rows",
            "If 051 receives meaningful safe-blend weight and improves LB/CV, consider GPU LogReg stacker add051 next",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add051_trimmed_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260615_052_blend_add051_trimmed_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_015.csv
 - pairwise_oof_correlation_focus_033.csv
 - pairwise_oof_correlation_focus_035.csv
 - pairwise_oof_correlation_focus_036_blend.csv
 - pairwise_oof_correlation_focus_038.csv
 - pairwise_oof_correlation_focus_039.csv
 - pairwise_oof_correlation_focus_040.csv
 - pairwise_oof_correlation_focus_045.csv
 - pairwise_oof_correlation_focus_049.csv
 - pairwise_oof_correlation_focus_050.csv
 - pairwise_oof_correlation_focus_051.csv
 - pred_exp_20260615_052_blend_add051_trimmed_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_015.csv
 - safe_blend_diagnostics_focus_026.csv
 - safe_blend_diagnostics_focus_028.csv
 - safe_blend_diagnostics_focus_029.csv
 - saf